# Power Market Dispatch & Fuel Switching Analytics

**Data source:** U.S. Energy Information Administration (EIA), 2025 EIA-923

### Objective
Analyze how natural gas and coal generation, plant efficiency, and delivered fuel economics varied during 2025.

### Questions
1. How did gas and coal generation change through the year?
2. What were the weighted average plant heat rates?
3. How did delivered gas and coal fuel costs vary?
4. Is there an observable association between relative fuel costs and the gas share of generation?

> **Important:** The regression is exploratory and measures association, not causation. It uses only 12 monthly observations.


## 1. Import libraries

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

ROOT = Path.cwd().resolve()

# If the notebook is opened from the notebooks/ folder,
# move one level up to the project root.
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
CHARTS = ROOT / "outputs" / "charts"

print("Project root:", ROOT)


## 2. Load the EIA-923 generation and fuel data

The first workbook contains plant-level generation and electric fuel-consumption information.

In [ ]:
generation_file = next(
    RAW.glob("EIA923_Schedules_2_3_4_5_M_12_2025_Final*.xlsx")
)

generation_raw = pd.read_excel(
    generation_file,
    sheet_name="Page 1 Generation and Fuel Data",
    header=5
)

print("Rows:", len(generation_raw))
print("Columns:", len(generation_raw.columns))

generation_raw.head(3)


## 3. Clean and define the analysis sample

For this portfolio project, the analysis uses:

- Natural gas (`NG`) and coal (`COL`)
- Non-CHP plants
- EIA sectors 1 and 2
- Positive generation and positive electric fuel consumption
- Heat-rate values between 3 and 25 MMBtu/MWh as a simple outlier screen

This keeps the analysis focused and makes the methodology easy to explain in an interview.


In [ ]:
df = generation_raw.copy()

df = df[df["MER Fuel Type Code"].isin(["NG", "COL"])].copy()
df = df[df["Combined Heat And Power Plant"].eq("N")].copy()
df = df[df["EIA Sector Number"].isin([1, 2])].copy()

df["fuel_group"] = df["MER Fuel Type Code"].map({
    "NG": "Natural Gas",
    "COL": "Coal"
})

df["Net Generation (Megawatthours)"] = pd.to_numeric(
    df["Net Generation (Megawatthours)"], errors="coerce"
)
df["Elec Fuel Consumption MMBtu"] = pd.to_numeric(
    df["Elec Fuel Consumption MMBtu"], errors="coerce"
)

df = df[
    (df["Net Generation (Megawatthours)"] > 0) &
    (df["Elec Fuel Consumption MMBtu"] > 0)
].copy()

df["heat_rate_mmbtu_mwh"] = (
    df["Elec Fuel Consumption MMBtu"] /
    df["Net Generation (Megawatthours)"]
)

df = df[df["heat_rate_mmbtu_mwh"].between(3, 25)].copy()

df = df.rename(columns={
    "Plant Id": "plant_id",
    "Plant Name": "plant_name",
    "Plant State": "state",
    "Net Generation (Megawatthours)": "net_generation_mwh",
    "Elec Fuel Consumption MMBtu": "electric_fuel_mmbtu"
})

plant = df[
    [
        "plant_id",
        "plant_name",
        "state",
        "fuel_group",
        "net_generation_mwh",
        "electric_fuel_mmbtu",
        "heat_rate_mmbtu_mwh"
    ]
].copy()

print("Records after cleaning:", len(plant))
plant.head()


## 4. Plant-level heat-rate analysis

Heat rate is calculated as:

**Heat rate = fuel energy consumed (MMBtu) / electricity generated (MWh)**

A lower heat rate means less fuel energy is required per MWh of electricity generated.

In [ ]:
weighted_heat_rate = (
    plant.groupby("fuel_group")
    .apply(
        lambda g: np.average(
            g["heat_rate_mmbtu_mwh"],
            weights=g["net_generation_mwh"]
        ),
        include_groups=False
    )
    .rename("weighted_heat_rate_mmbtu_mwh")
    .reset_index()
)

weighted_heat_rate


## 5. Monthly generation

The EIA-923 workbook reports separate monthly `Netgen` fields. We reshape those fields into a monthly dataset.

In [ ]:
month_columns = {
    "Jan": "Netgen_Jan",
    "Feb": "Netgen_Feb",
    "Mar": "Netgen_Mar",
    "Apr": "Netgen_Apr",
    "May": "Netgen_May",
    "Jun": "Netgen_Jun",
    "Jul": "Netgen_Jul",
    "Aug": "Netgen_Aug",
    "Sep": "Netgen_Sep",
    "Oct": "Netgen_Oct",
    "Nov": "Netgen_Nov",
    "Dec": "Netgen_Dec"
}

source = generation_raw.copy()
source = source[source["MER Fuel Type Code"].isin(["NG", "COL"])].copy()
source = source[source["Combined Heat And Power Plant"].eq("N")].copy()
source = source[source["EIA Sector Number"].isin([1, 2])].copy()

source["fuel_group"] = source["MER Fuel Type Code"].map({
    "NG": "Natural Gas",
    "COL": "Coal"
})

monthly_rows = []

for month_name, column in month_columns.items():
    temp = source[["fuel_group", column]].copy()
    temp["generation_mwh"] = pd.to_numeric(
        temp[column], errors="coerce"
    ).fillna(0)

    grouped = (
        temp.groupby("fuel_group")["generation_mwh"]
        .sum()
        .reset_index()
    )

    grouped["month"] = month_name
    grouped["month_number"] = list(month_columns).index(month_name) + 1
    monthly_rows.append(grouped)

monthly_generation_long = pd.concat(monthly_rows, ignore_index=True)

monthly_generation = (
    monthly_generation_long
    .pivot(
        index=["month_number", "month"],
        columns="fuel_group",
        values="generation_mwh"
    )
    .reset_index()
    .rename_axis(None, axis=1)
    .sort_values("month_number")
)

monthly_generation


In [ ]:
# Portfolio chart: monthly generation

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 14,
    "axes.labelsize": 10
})

x = np.arange(len(monthly_generation))
width = 0.36

fig, ax = plt.subplots(figsize=(10.5, 5.8))

ax.bar(
    x - width/2,
    monthly_generation["Natural Gas"] / 1e6,
    width,
    label="Natural gas",
    color="#2F6B9A"
)

ax.bar(
    x + width/2,
    monthly_generation["Coal"] / 1e6,
    width,
    label="Coal",
    color="#777777"
)

ax.set_xticks(x)
ax.set_xticklabels(monthly_generation["month"])
ax.set_ylabel("Net generation (million MWh)")
ax.set_title(
    "Monthly generation from natural gas and coal",
    loc="left",
    weight="bold"
)
ax.text(
    0, 1.015,
    "2025 | filtered EIA-923 sample",
    transform=ax.transAxes,
    fontsize=9,
    color="#666666"
)
ax.grid(axis="y", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncol=2)

plt.tight_layout()
plt.show()


## 6. Load fuel receipts and calculate delivered fuel cost

The EIA-923 fuel-receipts table contains fuel quantity, heat content, and reported fuel cost. The project calculates a monthly weighted average delivered cost.

**Caveat:** fuel receipts represent deliveries, so the delivered cost is not necessarily the exact same-month fuel cost of electricity generation.

In [ ]:
receipts_file = generation_file

receipts_raw = pd.read_excel(
    receipts_file,
    sheet_name="Page 5 Fuel Receipts and Costs",
    header=4
)

receipts = receipts_raw[
    receipts_raw["FUEL_GROUP"].isin(["Natural Gas", "Coal"])
].copy()

receipts["QUANTITY"] = pd.to_numeric(
    receipts["QUANTITY"], errors="coerce"
)
receipts["Average Heat Content"] = pd.to_numeric(
    receipts["Average Heat Content"], errors="coerce"
)
receipts["FUEL_COST"] = pd.to_numeric(
    receipts["FUEL_COST"], errors="coerce"
)

receipts["fuel_cost_usd_mmbtu"] = receipts["FUEL_COST"] / 100
receipts["received_mmbtu"] = (
    receipts["QUANTITY"] *
    receipts["Average Heat Content"]
)

receipts = receipts[
    (receipts["received_mmbtu"] > 0) &
    (receipts["fuel_cost_usd_mmbtu"] > 0)
].copy()

receipts["weighted_cost"] = (
    receipts["fuel_cost_usd_mmbtu"] *
    receipts["received_mmbtu"]
)

monthly_prices = (
    receipts
    .groupby(["YEAR", "MONTH", "FUEL_GROUP"], as_index=False)
    .agg(
        total_received_mmbtu=("received_mmbtu", "sum"),
        weighted_cost=("weighted_cost", "sum")
    )
)

monthly_prices["fuel_cost_usd_mmbtu"] = (
    monthly_prices["weighted_cost"] /
    monthly_prices["total_received_mmbtu"]
)

monthly_prices.head()


In [ ]:
price_pivot = (
    monthly_prices
    .pivot(
        index=["YEAR", "MONTH"],
        columns="FUEL_GROUP",
        values="fuel_cost_usd_mmbtu"
    )
    .reset_index()
    .rename(columns={
        "Natural Gas": "gas_price_usd_mmbtu",
        "Coal": "coal_price_usd_mmbtu"
    })
)

price_pivot["month_number"] = price_pivot["MONTH"]
price_pivot["month"] = pd.to_datetime(
    price_pivot["month_number"], format="%m"
).dt.strftime("%b")

price_pivot.head()


In [ ]:
# Portfolio chart: delivered fuel cost

fig, ax = plt.subplots(figsize=(10.5, 5.8))

ax.plot(
    price_pivot["month_number"],
    price_pivot["gas_price_usd_mmbtu"],
    marker="o",
    markersize=4.5,
    linewidth=2,
    label="Natural gas",
    color="#2F6B9A"
)

ax.plot(
    price_pivot["month_number"],
    price_pivot["coal_price_usd_mmbtu"],
    marker="o",
    markersize=4.5,
    linewidth=2,
    label="Coal",
    color="#777777"
)

ax.set_xticks(price_pivot["month_number"])
ax.set_xticklabels(price_pivot["month"])
ax.set_ylabel("Delivered fuel cost (USD/MMBtu)")
ax.set_title(
    "Monthly delivered fuel cost",
    loc="left",
    weight="bold"
)
ax.text(
    0, 1.015,
    "2025 | weighted by reported fuel receipts",
    transform=ax.transAxes,
    fontsize=9,
    color="#666666"
)
ax.grid(axis="y", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncol=2)

plt.tight_layout()
plt.show()


## 7. Combine generation and fuel economics

In [ ]:
market = monthly_generation.merge(
    price_pivot[
        [
            "month_number",
            "month",
            "gas_price_usd_mmbtu",
            "coal_price_usd_mmbtu"
        ]
    ],
    on=["month_number", "month"],
    how="inner"
)

market["total_gas_coal_generation"] = (
    market["Natural Gas"] +
    market["Coal"]
)

market["gas_generation_share"] = (
    market["Natural Gas"] /
    market["total_gas_coal_generation"]
)

market["coal_generation_share"] = (
    market["Coal"] /
    market["total_gas_coal_generation"]
)

market["gas_coal_price_ratio"] = (
    market["gas_price_usd_mmbtu"] /
    market["coal_price_usd_mmbtu"]
)

market


## 8. Fuel economics and generation mix

The next analysis asks whether months with a higher gas-to-coal delivered-cost ratio also had a different gas share of generation.

This is an **association**, not evidence that fuel price differences alone caused the generation change.

In [ ]:
X = sm.add_constant(
    market["gas_coal_price_ratio"]
)

y = market["gas_generation_share"]

model = sm.OLS(y, X).fit()

print(model.summary())


In [ ]:
# Portfolio chart: regression relationship

x_values = market["gas_coal_price_ratio"].to_numpy()
y_values = (market["gas_generation_share"] * 100).to_numpy()

line_coefficients = np.polyfit(x_values, y_values, 1)
line_x = np.linspace(
    x_values.min() * 0.97,
    x_values.max() * 1.03,
    100
)
line_y = (
    line_coefficients[0] * line_x +
    line_coefficients[1]
)

fig, ax = plt.subplots(figsize=(8.6, 6.0))

ax.scatter(
    x_values,
    y_values,
    s=48,
    facecolor="white",
    edgecolor="#2F6B9A",
    linewidth=1.6
)

ax.plot(
    line_x,
    line_y,
    color="#555555",
    linewidth=1.6
)

for i, month in enumerate(market["month"]):
    ax.annotate(
        month,
        (x_values[i], y_values[i]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
        color="#555555"
    )

ax.set_xlabel("Gas / coal delivered-cost ratio")
ax.set_ylabel(
    "Natural gas share of gas + coal generation (%)"
)
ax.set_title(
    "Fuel economics and generation mix",
    loc="left",
    weight="bold"
)
ax.text(
    0, 1.015,
    "2025 monthly observations | exploratory OLS association",
    transform=ax.transAxes,
    fontsize=9,
    color="#666666"
)
ax.grid(color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

ax.text(
    0.98, 0.06,
    f"OLS R² = {model.rsquared:.3f}\n"
    f"p-value = {model.pvalues['gas_coal_price_ratio']:.2e}\n"
    f"n = {int(model.nobs)}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=9,
    color="#444444"
)

plt.tight_layout()
plt.show()


## 9. Key results

The values below are calculated from the actual 2025 EIA-923 data used in this project.

In [ ]:
gas_generation = market["Natural Gas"].sum()
coal_generation = market["Coal"].sum()
total_generation = gas_generation + coal_generation

gas_weighted_hr = np.average(
    plant.loc[
        plant["fuel_group"].eq("Natural Gas"),
        "heat_rate_mmbtu_mwh"
    ],
    weights=plant.loc[
        plant["fuel_group"].eq("Natural Gas"),
        "net_generation_mwh"
    ]
)

coal_weighted_hr = np.average(
    plant.loc[
        plant["fuel_group"].eq("Coal"),
        "heat_rate_mmbtu_mwh"
    ],
    weights=plant.loc[
        plant["fuel_group"].eq("Coal"),
        "net_generation_mwh"
    ]
)

gas_avg_cost = np.average(
    monthly_prices.loc[
        monthly_prices["FUEL_GROUP"].eq("Natural Gas"),
        "fuel_cost_usd_mmbtu"
    ],
    weights=monthly_prices.loc[
        monthly_prices["FUEL_GROUP"].eq("Natural Gas"),
        "total_received_mmbtu"
    ]
)

coal_avg_cost = np.average(
    monthly_prices.loc[
        monthly_prices["FUEL_GROUP"].eq("Coal"),
        "fuel_cost_usd_mmbtu"
    ],
    weights=monthly_prices.loc[
        monthly_prices["FUEL_GROUP"].eq("Coal"),
        "total_received_mmbtu"
    ]
)

results = pd.DataFrame({
    "Metric": [
        "Records after cleaning",
        "Natural gas generation (million MWh)",
        "Coal generation (million MWh)",
        "Natural gas share",
        "Coal share",
        "Natural gas weighted heat rate",
        "Coal weighted heat rate",
        "Average delivered gas cost (USD/MMBtu)",
        "Average delivered coal cost (USD/MMBtu)",
        "Regression R²",
        "Regression p-value",
        "Regression observations"
    ],
    "Value": [
        len(plant),
        gas_generation / 1e6,
        coal_generation / 1e6,
        gas_generation / total_generation,
        coal_generation / total_generation,
        gas_weighted_hr,
        coal_weighted_hr,
        gas_avg_cost,
        coal_avg_cost,
        model.rsquared,
        model.pvalues["gas_coal_price_ratio"],
        int(model.nobs)
    ]
})

results


## 10. Interpretation

### What the analysis shows

- Natural gas accounted for about **63%** of gas + coal generation in the filtered 2025 sample.
- The weighted average heat rates were about **10.24 MMBtu/MWh for natural gas** and **10.66 MMBtu/MWh for coal**.
- Average delivered fuel costs were approximately **$4.08/MMBtu for natural gas** and **$2.49/MMBtu for coal**.
- The exploratory regression produced an **R² of about 0.913** using 12 monthly observations.

### Important limitation

The regression should **not** be presented as proof of causality. Other factors such as electricity demand, outages, weather, plant availability, renewable generation, and operational constraints can affect the generation mix.

The fuel-cost measure is based on **reported fuel receipts/deliveries**, which do not necessarily represent the exact fuel consumed in the same month.


## 11. Export the analysis datasets

These files can be used for Power BI or further SQL analysis.


In [ ]:
plant.to_csv(
    PROCESSED / "plant_annual.csv",
    index=False
)

monthly_generation.to_csv(
    PROCESSED / "monthly_generation_notebook.csv",
    index=False
)

monthly_prices.to_csv(
    PROCESSED / "monthly_fuel_prices.csv",
    index=False
)

market.to_csv(
    PROCESSED / "monthly_market_analysis.csv",
    index=False
)

print("Saved processed datasets to:", PROCESSED)
